In [3]:
import numpy as np
import os
from tqdm import tqdm

def inject_doppler_to_dataset(input_dir="./dataset_final_ready_v5/", 
                              output_dir="./dataset_doppler/", 
                              doppler_shifts=[100, 300, 500, 800, 1000], # 测试残余多普勒频偏
                              fs=200e3):
    """
    对已有的复基带 NPY 数据直接注入多普勒频移
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 找到所有的特征数据文件
    x_files = sorted([f for f in os.listdir(input_dir) if f.endswith('_X.npy')])

    if not x_files:
        print(f"❌ 警告：在 {input_dir} 中没有找到任何 _X.npy 文件，请检查路径！")
        return

    # 生成时间序列 t (针对 1024 个采样点)
    # 因为你的有效负载长度 L_pay 是 1024
    L_pay = 1024
    t = np.arange(L_pay) / fs

    for fd in doppler_shifts:
        # 【修改点 1】：打印提示时保留1位小数，显示更清晰
        print(f"\n🌀 正在生成多普勒频偏 {fd} Hz ({fd/1000:.1f} kHz) 的测试集...")
        
        # 计算该频偏下的旋转因子：exp(j * 2 * pi * fd * t)
        rotation_factor = np.exp(1j * 2 * np.pi * fd * t)
        
        # 【核心修复点】：去除 int() 取整，改为保留一位小数。
        # 这样会生成 fd_0.1kHz, fd_0.3kHz 等文件夹，完美区分不同的频偏！
        fd_dir = os.path.join(output_dir, f"fd_{fd/1000:.1f}kHz")
        os.makedirs(fd_dir, exist_ok=True)

        for x_file in tqdm(x_files):
            # 1. 加载原始纯净数据，形状为 (N, 1024, 2)
            X_real_imag = np.load(os.path.join(input_dir, x_file))
            
            # 同步拷贝对应的标签 Y 文件
            y_file = x_file.replace('_X.npy', '_Y.npy')
            Y_labels = np.load(os.path.join(input_dir, y_file))
            
            # 2. 将 [Real, Imag] 重建为复数数组 (N, 1024)
            X_complex = X_real_imag[..., 0] + 1j * X_real_imag[..., 1]
            
            # 3. 核心物理模拟：注入多普勒频移
            # 利用 numpy 的广播机制，每一帧都乘上旋转因子
            X_doppler_complex = X_complex * rotation_factor
            
            # 4. 再次拆分成 [Real, Imag] 格式，以适配你的神经网络输入
            X_doppler_real_imag = np.stack([X_doppler_complex.real, X_doppler_complex.imag], axis=-1)
            
            # 5. 保存带有频偏的新数据集
            np.save(os.path.join(fd_dir, x_file), X_doppler_real_imag.astype(np.float32))
            np.save(os.path.join(fd_dir, y_file), Y_labels) # 标签不变，直接原样保存

    print("\n✅ 所有多普勒测试集生成完毕！请去 dataset_doppler 目录下查看。")

if __name__ == "__main__":
    # 请确保 input_dir 是你上一步生成的干净数据集的路径
    inject_doppler_to_dataset(input_dir="/root/autodl-tmp/validate/0218/Perception/dataset_final_ready_v5/", 
                              output_dir="/root/autodl-tmp/validate/0218/Perception/dataset_doppler/",
                              doppler_shifts=[100, 300, 500, 800, 1000])


🌀 正在生成多普勒频偏 100 Hz (0.1 kHz) 的测试集...


100%|██████████| 18/18 [00:06<00:00,  2.98it/s]



🌀 正在生成多普勒频偏 300 Hz (0.3 kHz) 的测试集...


100%|██████████| 18/18 [00:06<00:00,  2.85it/s]



🌀 正在生成多普勒频偏 500 Hz (0.5 kHz) 的测试集...


...


🌀 正在生成多普勒频偏 800 Hz (0.8 kHz) 的测试集...


100%|██████████| 18/18 [00:06<00:00,  2.90it/s]



🌀 正在生成多普勒频偏 1000 Hz (1.0 kHz) 的测试集...


100%|██████████| 18/18 [00:06<00:00,  2.98it/s]


✅ 所有多普勒测试集生成完毕！请去 dataset_doppler 目录下查看。
